In [6]:
import numpy as np


def im2col_matrix_cross_dense(Xin, K, S=1, dtype=np.float32):
    N, Cin, Hin, Win = Xin.shape
    Hout, Wout = Hin - K + 1, Win - K + 1
    patch_size = Cin * (Win*K + Hin*K - K*K)

    im2col_mat = np.zeros((Cin * Hin * Win, Hout * Wout * patch_size), dtype=dtype)

    patch_id = 0
    for i in range(0, Hout, S):
        for j in range(0, Wout, S):
            # Build mask for the cross patch
            mask = np.zeros((Hin, Win), dtype=bool)
            mask[i:i+K, :] = True         # horizontal band
            mask[:, j:j+K] = True         # vertical band
            idx = np.where(mask.flatten())[0]

            for c in range(Cin):
                base_in = c * Hin * Win
                base_out = patch_id * patch_size + c * (Win*K + Hin*K - K*K)
                im2col_mat[base_in + idx, base_out:base_out + len(idx)] = np.eye(len(idx), dtype=dtype)

            patch_id += 1

    return im2col_mat



# Sanity check

In [4]:
Xin = np.arange(2*2*5*5).reshape(2,2,5,5)
print("Showing the first image, i.e., X[0]")
Xin[0]

Showing the first image, i.e., X[0]


array([[[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24]],

       [[25, 26, 27, 28, 29],
        [30, 31, 32, 33, 34],
        [35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44],
        [45, 46, 47, 48, 49]]])

In [7]:
N, Cin, Hin, Win = Xin.shape

K = 2
S = 1

Hout, Wout = Hin - K + 1, Win - K + 1
P = Hout * Wout

patch_size = Cin*(Win*K + Hin*K -  K * K) 

Xin_flat = Xin.reshape(-1, Cin * Hin * Win)

im2col_mat = im2col_matrix_cross_dense(Xin, K,S) 

Xin_im2col = Xin_flat @ im2col_mat

Xin_patches_flat = Xin_im2col.reshape(N, P, patch_size)
np.sort(Xin_patches_flat[0][6])

# Sanity check: this block should output
# array([ 2.,  3.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14., 17.,
#        18., 22., 23., 27., 28., 30., 31., 32., 33., 34., 35., 36., 37.,
#        38., 39., 42., 43., 47., 48.])

array([ 2.,  3.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14., 17.,
       18., 22., 23., 27., 28., 30., 31., 32., 33., 34., 35., 36., 37.,
       38., 39., 42., 43., 47., 48.])